In [140]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate



In [128]:
loader = PyPDFLoader("../data/deeplearningbook-ml.pdf")
pdf_data = loader.load()
print(pdf_data[0].page_content)



Chapter5MachineLearningBasics
Deeplearningisaspeciﬁckindofmachinelearning.Inordertounderstanddeeplearningwell,onemusthaveasolidunderstandingofthebasicprinciplesofmachinelearning.Thischapterprovidesabriefcourseinthemostimportantgeneralprinciplesthatwillbeappliedthroughouttherestofthebook.Novicereadersorthosewhowantawiderperspectiveareencouragedtoconsidermachinelearningtextbookswithamorecomprehensivecoverageofthefundamentals,suchasMurphy(2012)orBishop(2006).Ifyouarealreadyfamiliarwithmachinelearningbasics,feelfreetoskipaheadtoSec.5.11.Thatsectioncoverssomeper-spectivesontraditionalmachinelearningtechniquesthathavestronglyinﬂuencedthedevelopmentofdeeplearningalgorithms.Webeginwithadeﬁnitionofwhatalearningalgorithmis,andpresentanexample:thelinearregressionalgorithm. Wethenproceedtodescribehowthechallengeofﬁttingthetrainingdatadiﬀersfromthechallengeofﬁndingpatternsthatgeneralizetonewdata.Mostmachinelearningalgorithmshavesettingscalledhyperparametersthatmustbedeterminedexternaltothelearninga

In [129]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(pdf_data)
len(splitter_data)

215

In [130]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [131]:
vector_store = Chroma.from_documents(
    documents=splitter_data, 
    embedding=embeddings
)

In [132]:
query = "Explain Deep Learning ?"
data = vector_store.similarity_search(query=query, k=5)

print(data[0].page_content)

mPriortotheadventofdeeplearning,themainwaytolearnnonlinearmodelswastousethekerneltrickincombinationwithalinearmodel.Manykernellearningalgorithmsrequireconstructinganmm×matrixGi,j=k(x()i,x()j).ConstructingthismatrixhascomputationalcostO(m2),whichisclearlyundesirablefordatasetswith billionsof examples.In academia, starting in2006,deeplearning wasinitiallyinterestingbecauseitwasabletogeneralizetonewexamplesbetterthancompetingalgorithmswhentrainedonmedium-sizeddatasetswithtensofthousandsofexamples.Soonafter,deeplearninggarneredadditionalinterestinindustry,becauseitprovidedascalablewayoftrainingnonlinearmodelsonlargedatasets.StochasticgradientdescentandmanyenhancementstoitaredescribedfurtherinChapter8.5.10BuildingaMachineLearningAlgorithmNearlyalldeeplearningalgorithmscanbedescribedasparticularinstancesofafairlysimplerecipe:combineaspeciﬁcationofadataset,acostfunction,anoptimizationprocedureandamodel.Forexample,thelinearregressionalgorithmcombinesadatasetconsistingof153


In [133]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

In [134]:
llm = ChatOpenAI(model="gpt-5")

# result = llm.invoke(f"""can you provide me the answer based on 
# provided context for my question, context : {context}, question: {query}""")


In [135]:
def get_context(query:str):
    print(query)
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context": context,
        "query": query
    }

In [136]:
prompt = PromptTemplate.from_template("""
    your are a helpful assistent and provide answer based on the context for user question and if you don't know the answer based on the context then say "I don't know".
    Context : {context}
    Question : {query}
""")

In [137]:
rag_chain = get_context | prompt  | llm

In [138]:
res_data = rag_chain.invoke("explain Course?")

explain Course?


In [139]:
print(res_data.content)

I don't know. The context only discusses machine learning basics and a linear regression example (predicting a scalar y from a vector x with a linear function) and notes there’s no rigid taxonomy of datasets or experiences. It doesn’t define or describe any course.
